# 🏷️ Naive Bayes Classifier — Solutions Notebook

**Complete, verified solutions.**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('Setup complete! ✅')

### Gaussian Naive Bayes from Scratch Solution

In [ ]:
# ✅ SOLUTION: Gaussian Naive Bayes
class GaussianNaiveBayes:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        self.means = np.zeros((n_classes, n_features))
        self.vars = np.zeros((n_classes, n_features))
        self.priors = np.zeros(n_classes)
        
        for c_idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.means[c_idx, :] = X_c.mean(axis=0)
            self.vars[c_idx, :] = X_c.var(axis=0) + 1e-9  # Laplace-style epsilon smoothing
            self.priors[c_idx] = X_c.shape[0] / float(n_samples)

    def _pdf(self, class_idx, x):
        mean = self.means[class_idx]
        var = self.vars[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict(self, X):
        y_pred = [self._predict_sample(x) for x in X]
        return np.array(y_pred)

    def _predict_sample(self, x):
        posteriors = []
        for c_idx in range(len(self.classes)):
            prior = np.log(self.priors[c_idx])
            conditional = np.sum(np.log(self._pdf(c_idx, x)))
            posteriors.append(prior + conditional)
        return self.classes[np.argmax(posteriors)]

# Test implementation
X, y = make_classification(n_samples=500, n_features=4, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GaussianNaiveBayes()
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f'Accuracy from scratch: {accuracy_score(y_test, preds):.4f}')

### scikit-learn Naive Bayes Comparison & Text Classification

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

sk_gnb = GaussianNB()
sk_gnb.fit(X_train, y_train)
print(f'scikit-learn GaussianNB Accuracy: {sk_gnb.score(X_test, y_test):.4f}')

# MultinomialNB with Laplace Smoothing
corpus = [
    'Offer valid now win free money',
    'Win a brand new car today for free',
    'Meeting scheduled for tomorrow morning',
    'Project report submission deadline extended',
    'Free prize claim your reward instantly'
]
labels = [1, 1, 0, 0, 1]

vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(corpus)

mnb = MultinomialNB(alpha=1.0)  # alpha=1.0 is Laplace smoothing
mnb.fit(X_vec, labels)

test_email = ['claim your free reward today']
test_vec = vectorizer.transform(test_email)
prediction = mnb.predict(test_vec)
print(f'Test Email Prediction: {"Spam" if prediction[0] == 1 else "Ham"}')